In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

df = pd.read_csv('tableau_information_feat.csv')

labels_df = pd.read_csv("../data/train_data/labels.csv")                

print(f"Features (from laz): {len(df)}")
print(f"Labels (from csv):   {len(labels_df)}")

df['join_id'] = df['filename'].apply(
    lambda x: os.path.splitext(os.path.basename(x))[0]
)

labels_df['join_id'] = labels_df['filename'].apply(
    lambda x: os.path.splitext(os.path.basename(x))[0]
)

full_data = pd.merge(df, labels_df, on='join_id', how='inner')

print(f"Matched Trees: {len(full_data)}")

train_df, val_df = train_test_split(
    full_data, 
    test_size=0.2, 
    random_state=42, 
    stratify=full_data['species']
)

Features (from laz): 17707
Labels (from csv):   17707
Matched Trees: 17707


In [6]:
feature_cols = [
        "crown_entropy",
        "trunk_entropy",
        "crown_trunk_ratio",
        "crown_pts_count",
        "trunk_pts_count"]

In [7]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# 1. Prepare Data
target_col = 'species'

# Ensure feature_cols is defined (assuming you have it from previous steps)
# feature_cols = ['height', 'crown_ratio', 'stem_diameter', 'trunk_height', ...] 

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_val = val_df[feature_cols]
y_val = val_df[target_col]

# XGBoost requires integer labels (0, 1, 2...)
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_val_encoded = le.transform(y_val)

# 2. Define and Fit XGBoost Model
# use_label_encoder=False removes a warning in newer versions
# eval_metric='mlogloss' prevents another warning for multi-class classification
xgb_model = XGBClassifier(
    n_estimators=2000, 
    learning_rate=0.1,    # Step size shrinkage to prevent overfitting
    max_depth=7,          # Depth of tree (increase if underfitting)
    random_state=42,
    n_jobs=-1,            # Use all CPU cores
    use_label_encoder=False,
    eval_metric='mlogloss'
)

print("Training XGBoost...")
xgb_model.fit(X_train, y_train_encoded)
print("Training Complete.")

# 3. Predict & Evaluate
print("Précision sur les données de validation")
y_pred = xgb_model.predict(X_val)
accuracy = accuracy_score(y_val_encoded, y_pred)

print(f"\nModel Accuracy: {accuracy:.2%}")
print("-" * 30)
print("Classification Report:")
print(classification_report(y_val_encoded, y_pred, target_names=le.classes_))

Training XGBoost...


c:\Users\mathe\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [14:38:34] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Training Complete.
Précision sur les données de validation

Model Accuracy: 44.04%
------------------------------
Classification Report:
                       precision    recall  f1-score   support

           Abies_alba       0.25      0.08      0.12        24
       Acer_campestre       0.35      0.38      0.36       248
  Acer_pseudoplatanus       0.48      0.50      0.49       115
       Acer_saccharum       0.36      0.18      0.24        22
       Betula_pendula       0.33      0.28      0.30       147
     Carpinus_betulus       0.25      0.23      0.24       249
     Corylus_avellana       0.56      0.56      0.56         9
   Crataegus_monogyna       0.00      0.00      0.00        46
   Eucalyptus_miniata       0.62      0.70      0.66        56
   Euonymus_europaeus       0.14      0.10      0.11        21
      Fagus_sylvatica       0.41      0.49      0.44       496
Fraxinus_angustifolia       0.33      0.14      0.20        21
   Fraxinus_excelsior       0.18      0.08 